### SESSION A

#### SCENARIO
- You are deploying a delivery time prediction model for a food delivery startup. The data
science team trained the model in a Jupyter notebook using scikit-learn, and you need
to ship it as part of a production Flask service that will handle several hundred requests
per hour

- Question 1: 
Compare Pickle and Joblib as serialisation choices for this deployment context.
Which would you use and why? What risks does each carry when the model is loaded back in an
environment that differs from the one it was trained in?


#### Answer

- When deploying a machine learning model in production, both Pickle and Joblib can save and load trained models. However, they work differently.

- Pickle is Python's built-in serialization library and works well for small Python objects. Joblib is optimized for NumPy arrays and large machine learning models, making it faster and more memory-efficient for scikit-learn models.

- For a food delivery prediction model built using scikit-learn, Joblib is the better choice because it loads large numerical data more efficiently and reduces loading time in production.

- Both methods have one major risk: if the production environment has a different Python version or different library versions (such as scikit-learn), the saved model may fail to load or produce incorrect predictions. Therefore, the deployment environment should use the same versions as the training environment.

### SCENARIO
- You are working on a food delivery app that wants to auto-generate high-quality
promotional images of dishes using only a restaurant's text description and a reference
photo of the plated meal. The marketing team wants realistic, visually rich results.

- Question 2:
 A colleague suggests using a Variational Autoencoder (VAE), while another
recommends a Stable Diffusion-based model. Compare the two approaches for this
image-generation task — address their key architectural differences, output quality, and which is
more appropriate for a production marketing pipeline.

#### Answer

- A Variational Autoencoder (VAE) is a generative model that learns a compressed representation (latent space) of images and reconstructs them. It is fast and efficient but often generates blurry images because it focuses on reconstruction rather than high visual realism.

- Stable Diffusion is a diffusion-based generative model that gradually removes noise from random data to create highly detailed and realistic images. It supports text prompts, image conditioning, and produces much higher-quality outputs than VAEs.

- For a food delivery marketing application where realistic promotional images are required, Stable Diffusion is the better choice because it generates photorealistic and visually appealing images suitable for advertisements. Although it requires more computation, its output quality is significantly better for production marketing pipelines.

### SCENARIO
- You are developing an AI assistant for a food delivery platform. The assistant must
handle two distinct jobs: (1) answering open-ended questions such as 'What cuisines
are trending near me this weekend?', and (2) classifying incoming customer reviews as
Positive, Neutral, or Negative for the analytics dashboard

- Question 3: GPT-style decoder-only models and BERT-style encoder-only models are both
available to your team. Which architecture is better suited to each job, and what fundamental
property of each architecture makes it the right choice for that specific task?


#### Answer

- GPT is a decoder-only Transformer model designed for text generation. It predicts the next word in a sequence, making it ideal for open-ended conversations, content generation, and chatbots.

- BERT is an encoder-only Transformer model that understands the entire sentence using bidirectional context. It excels at classification, sentiment analysis, and question answering.

- In this scenario:

- GPT should be used to answer open-ended customer questions such as "What cuisines are trending near me this weekend?"
- BERT should be used to classify customer reviews into Positive, Neutral, or Negative because it is specifically designed for text understanding and classification.

### SCENARIO
- You are building a customer support bot for a food delivery platform. The bot must
handle complaint categories it has never been explicitly trained on — such as 'wrong
item delivered', 'packaging damaged', and 'missing cutlery'. Your team has only 3
annotated complaint–response pairs available per category

- Question 4: Explain the difference between zero-shot and few-shot prompting. Given the
constraints described, which strategy would you apply, and how would you structure the prompt
to maximise the quality of the bot's responses?


#### Answer

- Zero-shot prompting means the model performs a task using only instructions without any examples.

- Few-shot prompting provides a small number of example input-output pairs before the actual query so the model understands the desired response style and format.

- Since only three annotated complaint-response examples per category are available, few-shot prompting is the better approach. The prompt should first include the example complaint-response pairs and then ask the model to respond to the new complaint in the same style. This improves consistency and response quality.

### SCENARIO
- You are designing a LangChain-based food ordering assistant. A simpler alternative is a
single LLM call that takes the user's message and returns a response directly. Your team
is debating whether to build a full agent with tools or use a straightforward LLMChain.

- Question 5: Describe what distinguishes a LangChain agent from a simple LLM chain. Given that
your food ordering assistant must check live restaurant availability, call an API to calculate
estimated delivery time, and validate discount codes, justify which approach is more
appropriate — and identify two risks the agent approach introduces.



#### Answer

- An LLMChain executes a fixed sequence: it receives a prompt and returns a response without interacting with external tools.

- A LangChain Agent can reason about the user's request, decide which tool to use, call APIs, search databases, and combine multiple actions before generating a response.

- For the food ordering assistant, a LangChain Agent is more suitable because it needs to:

- Check live restaurant availability
- Calculate delivery time using an API
- Validate discount codes

- These tasks require external tool usage beyond simple text generation.

### SCENARIO
- You are optimising a food delivery chatbot that must accurately answer questions
about the platform's 500-page policy document and a menu catalogue that is updated
daily. Your team is evaluating two approaches: fine-tuning the underlying LLM on this
data, or implementing a Retrieval-Augmented Generation (RAG) pipeline.

- Question 6: Compare fine-tuning and RAG for this specific scenario. Address update frequency,
cost, hallucination risk, and data privacy. Which approach is more suitable for a daily-updated,
document-grounded Q&A bot, and what limitations does your recommended approach still
carry?



#### Answer

- Fine-tuning modifies the model's internal parameters by retraining it on new data. It is useful for learning new behaviors but is expensive, time-consuming, and difficult to update frequently.

- Retrieval-Augmented Generation (RAG) retrieves relevant information from external documents during inference and provides it to the model. It is easier to update because only the document database needs to be refreshed.

- However, RAG still depends on retrieval quality. If the retrieval system fails to find the correct documents, the generated answer may still be incomplete or inaccurate.

### SESSION B

#### Task 1: Save & Load a Delivery Time Predictor

Train a simple scikit-learn regression model on dummy delivery data, serialise it to disk with
Joblib, reload it, and confirm the saved and reloaded model produce identical predictions.
(Foundational — single Module 13 concept: model serialisation and reload.)

- Generate a small dummy dataset (100 rows minimum) with features distance_km,
num_items, and rain_flag, and a target column delivery_time_min; split it 80/20 for training
and testing.
- Train a LinearRegression model and print its RMSE on the test split before saving.
- Save the trained model to delivery_model.joblib and print a confirmation message that
includes the file size in KB.
- Reload the model from disk, run a prediction on one new sample, and assert that the reloaded
model's output matches the original model's output for the same input — printing PASS or FAIL
accordingly.

In [26]:
import numpy as np
import pandas as pd
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [27]:
np.random.seed(42)

n = 100

distance_km = np.random.uniform(1, 20, n)

num_items = np.random.randint(1, 10, n)

rain_flag = np.random.randint(0, 2, n)

delivery_time_min = (
    10
    + distance_km * 3
    + num_items * 2
    + rain_flag * 8
    + np.random.normal(0, 2, n)
)

df = pd.DataFrame({
    "distance_km": distance_km,
    "num_items": num_items,
    "rain_flag": rain_flag,
    "delivery_time_min": delivery_time_min
})

print(df.head())

   distance_km  num_items  rain_flag  delivery_time_min
0     8.116262          7          1          54.207002
1    19.063572          1          0          70.155660
2    14.907885          4          1          70.276729
3    12.374511          4          0          56.551535
4     3.964354          5          1          40.839538


In [28]:
X = df[["distance_km", "num_items", "rain_flag"]]

y = df["delivery_time_min"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [29]:
model = LinearRegression()

model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [30]:
prediction = model.predict(X_test)

In [31]:
rmse = root_mean_squared_error(y_test, prediction)

print("\nRMSE :", round(rmse,2))


RMSE : 2.44


In [32]:
joblib.dump(model, "delivery_model.joblib")

file_size = os.path.getsize("delivery_model.joblib") / 1024

print(f"\nModel Saved Successfully!")

print(f"File Size : {file_size:.2f} KB")


Model Saved Successfully!
File Size : 0.86 KB


In [33]:
loaded_model = joblib.load("delivery_model.joblib")

print("\nModel Loaded Successfully!")


Model Loaded Successfully!


In [34]:
sample = [[5,3,1]]

original_prediction = model.predict(sample)

loaded_prediction = loaded_model.predict(sample)

print("\nOriginal Prediction :", original_prediction[0])

print("Loaded Prediction :", loaded_prediction[0])


Original Prediction : 38.95280486430337
Loaded Prediction : 38.95280486430337


c:\Users\tejal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\tejal\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [35]:
if np.allclose(original_prediction, loaded_prediction):

    print("\nPASS : Predictions Match")

else:

    print("\nFAIL : Predictions Do Not Match")


PASS : Predictions Match


#### Task 2: Flask REST API for Order Predictions

Wrap the saved delivery time model in a Flask REST API so a mobile app or front-end can
request live delivery estimates via HTTP. (Applied — Module 13: model loading combined with
Flask API design.)

- Load delivery_model.joblib once at application startup — not inside the route handler — and
keep it in memory for the lifetime of the process.
- Expose a POST endpoint at /predict that accepts a JSON body with the fields distance_km,
num_items, and rain_flag.
- Return a JSON response containing predicted_delivery_time_min rounded to one decimal
place and an HTTP 200 status code on success.
- Return a descriptive JSON error body with HTTP 400 if any required field is missing from the
request, without crashing the server — demonstrate this by testing with an incomplete
payload using a curl command or Postman screenshot.

In [49]:
from flask import Flask, request, jsonify
import joblib
import numpy as np

In [50]:
app = Flask(__name__)

In [51]:
model = joblib.load("delivery_model.joblib")

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [52]:
@app.route("/")
def home():
    return "Delivery Prediction API is Running"

In [53]:
@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    required_fields = [
        "distance_km",
        "num_items",
        "rain_flag"
    ]

    # Check Missing Fields
    for field in required_fields:

        if field not in data:
            return jsonify({
                "error": f"Missing required field: {field}"
            }), 400

    # Convert Input
    features = np.array([[
        data["distance_km"],
        data["num_items"],
        data["rain_flag"]
    ]])

    prediction = model.predict(features)

    return jsonify({
        "predicted_delivery_time_min": round(float(prediction[0]), 1)
    }), 200

In [54]:
if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1